# 故障恢复微实验：检查点为什么不能消除重复写入？

约 20–30 分钟。使用纯 Python 标准库、CPU；无需 API 密钥或安装依赖。按顺序运行单元格。

本实验用内存对象模拟远端业务服务与持久检查点：模拟进程崩溃时保留两者，恢复时重新创建执行过程。**不是完整 Agent 或数据库持久化实现，也不证明跨系统 exactly-once。**

先预测：远端已提交、本地尚未保存完成状态时，盲目重放会发生什么？

比较三种协议：盲目重放、稳定操作 ID 的幂等重放、先查询再幂等重试。观察业务提交次数、网络写请求次数和恢复状态。

对应专题：[Harness](https://suguilin.github.io/study/?id=lesson-harness) / [工具可靠性](https://suguilin.github.io/study/?id=lesson-tool-reliability)。

In [ ]:
from dataclasses import dataclass, field
import json
from pathlib import Path

class SimulatedCrash(Exception):
    pass

@dataclass
class RemoteService:
    idempotent: bool
    query_available: bool = True
    write_requests: int = 0
    commits: list = field(default_factory=list)
    ledger: dict = field(default_factory=dict)

    def write(self, operation_id, payload):
        self.write_requests += 1
        if self.idempotent and operation_id in self.ledger:
            previous = self.ledger[operation_id]
            if previous['payload'] != payload:
                raise ValueError('The same operation ID cannot change its payload')
            return previous
        result = {'operation_id': operation_id, 'payload': payload, 'receipt': len(self.commits) + 1}
        self.commits.append(result)
        self.ledger[operation_id] = result
        return result

    def query(self, operation_id):
        if not self.query_available:
            raise ConnectionError('Query is unavailable; absence has not been established')
        return self.ledger.get(operation_id)

def first_attempt(checkpoint, remote, crash_at):
    checkpoint.update(status='pending', operation_id='task-42-write-1', payload={'artifact': 'note-42'})
    if crash_at == 'before_submit':
        raise SimulatedCrash()
    receipt = remote.write(checkpoint['operation_id'], checkpoint['payload'])
    if crash_at == 'after_remote_commit':
        raise SimulatedCrash()
    checkpoint.update(status='confirmed', receipt=receipt)
    if crash_at == 'after_checkpoint':
        raise SimulatedCrash()

def recover(checkpoint, remote, protocol, reuse_id=True):
    if checkpoint['status'] == 'confirmed':
        return 'confirmed'
    operation_id = checkpoint['operation_id'] if reuse_id else 'new-id-on-retry'
    receipt = None
    if protocol == 'query_then_retry':
        try:
            receipt = remote.query(operation_id)
        except ConnectionError:
            checkpoint['status'] = 'unknown'
            return 'unknown'
    if receipt is None:
        receipt = remote.write(operation_id, checkpoint['payload'])
    checkpoint.update(status='confirmed', receipt=receipt)
    return 'confirmed'

def trial(protocol, crash_at, reuse_id=True, query_available=True):
    remote = RemoteService(idempotent=protocol != 'naive_replay', query_available=query_available)
    checkpoint = {}
    try:
        first_attempt(checkpoint, remote, crash_at)
    except SimulatedCrash:
        pass
    status = recover(checkpoint, remote, protocol, reuse_id)
    return {'protocol': protocol, 'crash_at': crash_at, 'status': status,
            'business_commits': len(remote.commits), 'write_requests': remote.write_requests}


## 1 · 改变崩溃位置

先看 `after_remote_commit`：本地检查点仍是 pending，远端已经提交。运行后比较 **业务提交次数**，再比较请求次数。两个指标不能混淆。

In [ ]:
protocols = ['naive_replay', 'idempotent_replay', 'query_then_retry']
faults = ['before_submit', 'after_remote_commit', 'after_checkpoint']
report = [trial(protocol, fault) for protocol in protocols for fault in faults]
print(f"{'protocol':<20} {'crash_at':<22} {'status':<10} commits  requests")
for row in report:
    print(f"{row['protocol']:<20} {row['crash_at']:<22} {row['status']:<10} {row['business_commits']:>7} {row['write_requests']:>9}")
assert trial('naive_replay', 'after_remote_commit')['business_commits'] == 2
assert all(row['business_commits'] == 1 for row in report if row['protocol'] != 'naive_replay')
assert trial('idempotent_replay', 'after_remote_commit')['write_requests'] == 2
assert trial('query_then_retry', 'after_remote_commit')['write_requests'] == 1


## 2 · 故意破坏协议

保持远端幂等实现不变，只在重试时换一个操作 ID。预测业务提交次数，再运行。接着模拟查询不可用：此时应保留 unknown，不能把无法查询当作“明确未提交”。

In [ ]:
changed_id = trial('idempotent_replay', 'after_remote_commit', reuse_id=False)
unavailable = trial('query_then_retry', 'after_remote_commit', query_available=False)
print('Changed retry identity:', changed_id)
print('Query unavailable:', unavailable)
assert changed_id['business_commits'] == 2
assert unavailable['status'] == 'unknown'
assert unavailable['business_commits'] == 1


## 3 · 留下你的理解

填写下面的回答并运行，可在 Colab 文件侧栏下载 `recovery_report.json`，也可将解释复制到网站练习或学习日志。

1. 检查点恢复了什么，为什么仍不能确认远端业务状态？
2. 稳定 ID、原子幂等记录和状态查询分别负责什么？本模拟省略了哪个原子性条件？
3. 将远端查询改为最终一致的返回时，query-then-retry 还需要哪些约束？

建议小改动：增加随机故障位置；把模拟的 ledger 换成有事务与唯一约束的 SQLite；观察边界，避免仅追求打印“成功”。

In [ ]:
my_explanation = ""  # Replace with your own reasoning.
export = {'experiment': 'durable-agent', 'data_scope': 'synthetic in-memory simulation',
          'results': report, 'changed_id': changed_id, 'query_unavailable': unavailable,
          'my_explanation': my_explanation}
Path('recovery_report.json').write_text(json.dumps(export, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved recovery_report.json. Download it from the Colab Files panel if you want to keep the result.')
